# Урок 18: Защита на AI агенти с криптографски разписки

## Практическа тетрадка

Тази тетрадка преминава през четири упражнения:

1. **Подпишете първата си разписка** за повикване на агентски инструмент и я проверете.
2. **Манипулирайте разписката** и наблюдавайте как проверката се проваля.
3. **Изградете верига от три разписки** и потвърдете целостта на веригата.
4. **Опаковайте повикване на инструмент от Microsoft Agent Framework**, така че всяко действие да издава разписка.

Всички криптографски примитиви са импортирани от добре поддържани библиотеки (`pynacl` за Ed25519, `jcs` за RFC 8785 каноничен JSON, `hashlib` от стандартната библиотека на Python за SHA-256). Логиката на разписката сама по себе си е обикновен Python, който можете да четете и модифицирате.

Изпълнявайте клетките по ред. Всяка секция е кратка и самостоятелна.


## Setup

Инсталирайте двете зависимости. И двете имат разрешаващи лицензи (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Помощни утилити

Тези двама помощници обработват base64url кодиране (без подравняване) и SHA-256 хеширане на произволни обекти. Те държат останалата част от тетрадката съсредоточена върху самата логика за разписките.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Подпишете първото си разписване

Представете си, че нашият агент за **Contoso Travel** току-що е потърсил полети от Сидни до Лос Анджелис за клиент. Искаме да запишем това извикване на инструмент като подписано разписване, за да може бъдещ одитор да го провери без да ни се доверява.

### Стъпка 1.1: Генериране на ключ за подписване

В производствена среда ключът за подписване на агента ще се съхранява в хардуерен модул за сигурност (HSM), Azure Key Vault или подобно защитено хранилище. За този урок генерираме нов ключ в паметта.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Стъпка 1.2: Създаване на пратката за разписката

Пратката съдържа всичко, което искаме разписката да удостоверява: кой е действал, с какъв инструмент, с какви аргументи, какво е върнато, под каква политика и кога. Хешираме аргументите и резултата, вместо да ги включваме на място, за да не се излага чувствително съдържание в разписката.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Стъпка 1.3: Подписване и сглобяване на квитанцията

Три стъпки:

1. Канонизирайте натоварването с помощта на JCS, така че две реализации, произвеждащи една и съща логическа квитанция, да произвеждат идентични байтове.
2. Хаширайте канонизирания байтов поток с SHA-256.
3. Подпишете хеша с частния ключ Ed25519.

Подписът се прикачва към оригиналното натоварване, за да се получи крайната квитанция.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Стъпка 1.4: Проверка на разписката

Проверката обръща процеса. Премахваме подписа, преизчисляваме каноничния хеш и проверяваме подписа спрямо публичния ключ в разписката.

Одитор, който извършва тази проверка, не се нуждае от нищо от нас освен самата разписка. Няма нужда да извиква услуга, да прави запитвания в директория на ключове или да разчита на доверие.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Трябва да видите `Receipt is valid: True`. Агентът е генерирал своя първи криптографски подписан запис за одит.


## Раздел 2: Манипулиране с касовата бележка

Целта на касовите бележки е да са устойчиви на манипулации. Нека го докажем.

Ще променим точно един символ в касовата бележка и ще наблюдаваме как проверката се проваля.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Какво току-що се случи?

Когато променихме `policy_id`, каноничните байтове се промениха. SHA-256 хешът на тези байтове се промени. Подписът (който беше над оригиналния хеш) вече не съвпада с новия хеш. Проверката правилно връща `False`.

Няма начин да се промени някое поле на разписката и тя все пак да бъде проверена, освен ако атакуващият не разполага с частния ключ. Докато частният ключ се съхранява в ключов склад и публичният ключ е публикуван, манипулацията е невъзможно да бъде скрита.

Опитайте сами: променете `tool_name` или `agent_id` или `timestamp` в клетката по-горе и я изпълнете отново. Всяка промяна произвежда невалидна разписка.


## Section 3: Свързване на разписките в последователност

Една разписка защитава едно действие. Повечето агенти извършват много действия. За да направим цялата последователност защитена от манипулации, свързваме всяка разписка с предишната, като включваме хеша на предишната разписка в полезния товар на новата разписка.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Ако някой премахне или промени реда на разписка, веригата се прекъсва точно на това място. Проверката на която и да е по-късна разписка се проваля, защото `previous_receipt_hash` вече не съвпада с действителния хеш на предишната.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Сега прекъснете веригата, като манипулирате средния разпис и повторно проверите. Манипулираният разпис не минава проверката на подписа си, И следващият разпис не минава проверката на връзката във веригата (защото неговият `previous_receipt_hash` вече не съвпада с хеша на модифицирания среден разпис).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Касова бележка 0 все още се проверява (тя не е променяна и няма предшественик, от който да зависи). Касова бележка 1 не минава проверката на подписа, защото променихме `tool_args_hash`. Касова бележка 2 не минава проверката на връзката в веригата, защото `previous_receipt_hash` е изчислен спрямо оригиналната (сега променена) касова бележка 1.

Дори ако нападателят преподпише променената касова бележка 1 (което не може да направи без частния ключ), несъвпадението във връзката на веригата в касова бележка 2 все пак би разкрило манипулацията. За да скрие промяната, нападателят трябва да преподпише всяка касова бележка от точката на промяната нататък, което изисква притежаване на частния ключ.


## Section 4: Обвийте обаждане към инструмент на агент с подписване на разписка

В реална среда за внедряване не искате всеки автор на агент да помни да извиква `make_receipt`. Искате подписването на разписки да бъде автоматично за всяко извикване на инструмент.

Ето най-простия модел: обвиващ клас, който приема всяка извикваема функция на инструмент и връща версия, която издава разписки. Това се адаптира към всяка рамка за агенти, включително Microsoft Agent Framework (`agent_framework.azure`).

Ако нямате настроен проект Azure AI Foundry, локалният макет по-долу все още демонстрира модела.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Интегриране с Microsoft Agent Framework

Обвивката `ReceiptedTool` по-горе е независима от рамката. За да я използвате в агент, създаден с Microsoft Agent Framework, регистрирайте обвитата функция като инструмент. Ето скица (ще замените макета с реална регистрация на инструмент в Azure AI Foundry):

```python
# Псевдокод, показващ формата на интеграция.
# от agent_framework.azure импортиране на AzureAIProjectAgentProvider
# от azure.identity импортиране на AzureCliCredential
#
# доставчик = AzureAIProjectAgentProvider(креденциали=AzureCliCredential())
# агент = доставчик.create_agent(
#     инструкции="Вие сте агент на Contoso Travel ...",
#     инструменти=[receipted_lookup],   # обвитият инструмент, не суровата функция
# )
# отговор = агент.run("Намерете полети от Сидни до Лос Анджелис през юни.")
#
# # След изпълнението, всеки инструментален извикване от агента има подписан разписка:
# audit_chain = receipted_lookup.receipts
```

Agent framework не трябва да знае нищо за разписките. Подписването на разписки е обвито около инструмента, а не вградено в рамката. Това е начинът да добавите произход към съществуващ код на агент без да преработвате агента.


## Обзор и допълнително предизвикателство

Вие:

- Генерирахте ключова двойка Ed25519.
- Създадохте и подписахте бележка за повикване на агентски инструмент.
- Проверихте бележката офлайн, използвайки само публичния ключ.
- Манипулирахте бележка и наблюдавахте неуспех при проверката.
- Създадохте последователност от три бележки, свързани чрез хеш верига.
- Манипулирахте с бележката в средата на веригата и наблюдавахте както провал на подписа, така и провал на свързването в веригата.
- Опаковахте функция на инструмент с автоматично подписване на бележка.

**Допълнително предизвикателство.** Разширете схемата на бележката с поле `request_id` (UUID за дистрибутирано проследяване). Актуализирайте `make_receipt`, за да го включва, и потвърдете, че бележките все още се проверяват от край до край. След това модифицирайте полето след подписването и потвърдете, че проверката не минава. Това ви принуждава да разберете как всеки байт от каноничното кодиране влияе върху подписа.

**Важно ограничение.** Бележките доказват три неща и само три неща: атрибуция (този ключ е подписал това съдържание), цялостност (съдържанието не се е променило след подписването) и подреждане (тази бележка е дошла след онази бележка). Те НЕ доказват, че действието на агента е било правилно, че правилото, посочено в `policy_id`, действително е било оценено, или че агентът е спазвал всяко правило. Бележките са основа. Управлението е системата, която изграждате върху нея.

Прочетете отново README на урока със съзнанието за това ограничение. Най-честата грешка, която правят екипите с бележките, е да приемат, че „имаме бележки“ означава „ние сме управлявани“. Това не е така. Бележките правят поведението на агента проверимо. Те не го правят правилно.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
